### Import Dependencies

In [1]:
import openai
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from langsmith import Client

from openai import AsyncOpenAI

from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics.collections import Faithfulness, AnswerRelevancy
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall

load_dotenv("../../.env", override=True)

/Users/codyliu/Documents/GitHub/ai-engineering-prerequisites/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/f4/ld0c514j5wgbqbdjsyrdc5v40000gn/T/ipykernel_36507/1069460277.py:12: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall
/var/folders/f4/ld0c514j5wgbqbdjsyrdc5v40000gn/T/ipykernel_36507/1069460277.py:12: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedConte

True

### Download an example reference data point from LangSmith

In [2]:
ls_client = Client()
dataset = ls_client.read_dataset(dataset_name="rag-evaluation-dataset")
examples = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))
len(examples)

34

In [3]:
# The course uses examples[25], but the questions in this dataset are different.
# Use an example that has reference product IDs, so that recall can be calculated.
example = next(e for e in examples if "WisaKey" in e.inputs["question"])

reference_input = example.inputs
reference_output = example.outputs
reference_input

{'question': 'Does the WisaKey ring light set come with a phone holder and tripod for TikTok videos?'}

In [4]:
reference_output["reference_context_ids"]

['B09QGRRY7G']

### RAG Pipeline

In [5]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding


def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings,
    }


def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(
        context["retrieved_context_ids"],
        context["retrieved_context"],
        context["retrieved_context_ratings"],
    ):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context


def build_prompt(preprocessed_context, query):
    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{query}
"""
    return prompt


def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{"role": "system", "content": prompt}],
        reasoning_effort="none",
    )
    return response.choices[0].message.content


def rag_pipeline(query, qdrant_client, top_k=5):
    retrieved_context = retrieve_data(query, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    answer = generate_answer(prompt)

    return {
        "answer": answer,
        "question": query,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
    }

In [6]:
result = rag_pipeline(reference_input["question"], qdrant_client)
print(result["answer"])
print()
print("Retrieved IDs:", result["retrieved_context_ids"])

Yes. The WisaKey 13" RGB Ring Light set includes 2 phone holders and a tripod stand (a floor tripod that can extend to 63" and a desktop tripod adjustable to 7.08"), which are suitable for TikTok videos.

Retrieved IDs: ['B09QGRRY7G', 'B09R1S39N5', 'B0BB6TFQ22', 'B0B8ZMQ53J', 'B0B76GG8L1']


### RAGAS Metrics

In [7]:
openai_client = AsyncOpenAI()

ragas_llm = llm_factory("gpt-5.4-mini", client=openai_client, max_completion_tokens=4000)

args = ragas_llm.model_args
args.pop("max_tokens", None)
args.pop("top_p", None)
args["temperature"] = 0.0
args["max_completion_tokens"] = 4000

ragas_embeddings = embedding_factory("openai", model="text-embedding-3-small", client=openai_client)

In [8]:
async def ragas_faithfulness(run, example):
    scorer = Faithfulness(llm=ragas_llm)
    return await scorer.ascore(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"],
    )


async def ragas_relevancy(run, example):
    scorer = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    return await scorer.ascore(
        user_input=run["question"],
        response=run["answer"],
    )


async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    return await IDBasedContextPrecision().single_turn_ascore(sample)


async def ragas_context_recall_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    return await IDBasedContextRecall().single_turn_ascore(sample)

In [9]:
print("Faithfulness:      ", await ragas_faithfulness(result, reference_output))
print("Answer relevancy:  ", await ragas_relevancy(result, reference_output))
print("Context precision: ", await ragas_context_precision_id_based(result, reference_output))
print("Context recall:    ", await ragas_context_recall_id_based(result, reference_output))

Faithfulness:       MetricResult(value=0.8)
Answer relevancy:   MetricResult(value=0.6660639090458204)
Context precision:  0.2
Context recall:     1.0
